In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

import torch
import yaml
import numpy as np
import glob
import cv2
from pathlib import Path
from collections import defaultdict, Counter
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

PROJECT_ROOT = '/mnt/Data/AKIB/YOLO-OD-IM'
DATA_YAML    = os.path.join(PROJECT_ROOT, 'dataset', 'data_abs.yaml')

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
class_names = cfg['names']
nc = cfg['nc']

# Test image paths
test_img_dir = os.path.join(PROJECT_ROOT, 'dataset', 'test', 'images')
test_lbl_dir = os.path.join(PROJECT_ROOT, 'dataset', 'test', 'labels')
test_images = sorted(glob.glob(os.path.join(test_img_dir, '*')))

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Test images: {len(test_images)}")
print(f"Classes: {nc}")

## 3.1 Load Best Models

In [ ]:
# ── Load the best available models ──
models = {}
model_paths = {
    'yolo11n': os.path.join(PROJECT_ROOT, 'runs', 'baseline_yolo11n', 'weights', 'best.pt'),
    'yolo11m': os.path.join(PROJECT_ROOT, 'runs', 'optimized_yolo11m_1280', 'weights', 'best.pt'),
    'yolo11l': os.path.join(PROJECT_ROOT, 'runs', 'optimized_yolo11l_1280', 'weights', 'best.pt'),
}

for name, path in model_paths.items():
    if os.path.exists(path):
        models[name] = YOLO(path)
        print(f"✅ Loaded {name}: {path}")
    else:
        print(f"⚠️  {name} not found: {path}")

# Pick the best single model available
best_model_name = list(models.keys())[-1]  # prefer largest
best_model = models[best_model_name]
print(f"\nBest single model: {best_model_name}")

## 3.2 Test-Time Augmentation (TTA)

YOLO's built-in TTA runs multiple inference passes with flips and scales, then merges predictions.

In [ ]:
# ── TTA Evaluation ──
print("Evaluating WITHOUT TTA...")
m_no_tta = best_model.val(
    data=DATA_YAML, split='test', device=0,
    conf=0.15, iou=0.6, verbose=False
)

print("\nEvaluating WITH TTA...")
m_tta = best_model.val(
    data=DATA_YAML, split='test', device=0,
    conf=0.15, iou=0.6, augment=True, verbose=False
)

print(f"\n{'':30s} {'Precision':>10s} {'Recall':>10s} {'mAP@50':>10s}")
print(f"{'Without TTA':30s} {m_no_tta.box.mp:10.4f} {m_no_tta.box.mr:10.4f} {m_no_tta.box.map50:10.4f}")
print(f"{'With TTA':30s} {m_tta.box.mp:10.4f} {m_tta.box.mr:10.4f} {m_tta.box.map50:10.4f}")
print(f"\nRecall improvement from TTA: {(m_tta.box.mr - m_no_tta.box.mr)*100:+.2f}%")

## 3.3 SAHI – Slicing Aided Hyper Inference

SAHI slices the image into overlapping tiles, runs detection on each, then merges results.
This is excellent for crowded shelf images with many small products.

In [ ]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction, get_prediction
from sahi.utils.yolov8 import download_yolov8s_model

# ── SAHI Inference ──
# Find best model path
best_path = model_paths[best_model_name]

detection_model = AutoDetectionModel.from_pretrained(
    model_type='ultralytics',
    model_path=best_path,
    confidence_threshold=0.15,
    device='cuda:0',
)

print("SAHI model loaded successfully.")
print(f"Model: {best_model_name}")

In [ ]:
# ── Run SAHI on test images and compare with standard inference ──
def parse_ground_truth(label_path, img_w, img_h):
    """Parse YOLO format labels to absolute bbox format."""
    gt_boxes = []
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    cx, cy, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                    x1 = (cx - w/2) * img_w
                    y1 = (cy - h/2) * img_h
                    x2 = (cx + w/2) * img_w
                    y2 = (cy + h/2) * img_h
                    gt_boxes.append({'class_id': cls_id, 'bbox': [x1, y1, x2, y2]})
    return gt_boxes

def compute_recall_precision(predictions, ground_truths, iou_threshold=0.5):
    """Compute recall and precision from predictions vs ground truth."""
    if len(ground_truths) == 0:
        return 1.0 if len(predictions) == 0 else 0.0, 1.0 if len(predictions) == 0 else 0.0
    if len(predictions) == 0:
        return 0.0, 1.0
    
    matched_gt = set()
    tp = 0
    
    for pred in predictions:
        best_iou = 0
        best_gt_idx = -1
        for i, gt in enumerate(ground_truths):
            if i in matched_gt:
                continue
            if pred['class_id'] != gt['class_id']:
                continue
            iou = compute_iou(pred['bbox'], gt['bbox'])
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = i
        if best_iou >= iou_threshold and best_gt_idx >= 0:
            tp += 1
            matched_gt.add(best_gt_idx)
    
    precision = tp / len(predictions) if len(predictions) > 0 else 0
    recall = tp / len(ground_truths) if len(ground_truths) > 0 else 0
    return recall, precision

def compute_iou(box1, box2):
    """Compute IoU between two boxes [x1, y1, x2, y2]."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    inter = max(0, x2-x1) * max(0, y2-y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

# Run SAHI vs standard
sahi_recalls, sahi_precisions = [], []
std_recalls, std_precisions = [], []

for img_path in test_images:
    img = cv2.imread(img_path)
    img_h, img_w = img.shape[:2]
    
    # Ground truth
    lbl_name = os.path.splitext(os.path.basename(img_path))[0] + '.txt'
    lbl_path = os.path.join(test_lbl_dir, lbl_name)
    gt = parse_ground_truth(lbl_path, img_w, img_h)
    
    # Standard inference
    std_result = get_prediction(img_path, detection_model)
    std_preds = [{
        'class_id': p.category.id,
        'bbox': p.bbox.to_xyxy(),
        'score': p.score.value
    } for p in std_result.object_prediction_list]
    
    # SAHI sliced inference
    sahi_result = get_sliced_prediction(
        img_path,
        detection_model,
        slice_height=320,
        slice_width=320,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
        perform_standard_pred=True,  # also run full-image prediction and merge
        postprocess_type='NMS',
        postprocess_match_threshold=0.5,
    )
    sahi_preds = [{
        'class_id': p.category.id,
        'bbox': p.bbox.to_xyxy(),
        'score': p.score.value
    } for p in sahi_result.object_prediction_list]
    
    # Compute metrics
    r_std, p_std = compute_recall_precision(std_preds, gt)
    r_sahi, p_sahi = compute_recall_precision(sahi_preds, gt)
    
    std_recalls.append(r_std)
    std_precisions.append(p_std)
    sahi_recalls.append(r_sahi)
    sahi_precisions.append(p_sahi)

print(f"\n{'Method':20s} {'Avg Recall':>12s} {'Avg Precision':>14s}")
print(f"{'Standard':20s} {np.mean(std_recalls):12.4f} {np.mean(std_precisions):14.4f}")
print(f"{'SAHI (320px tiles)':20s} {np.mean(sahi_recalls):12.4f} {np.mean(sahi_precisions):14.4f}")
print(f"\nSAHI recall improvement: {(np.mean(sahi_recalls) - np.mean(std_recalls))*100:+.2f}%")

## 3.4 Weighted Boxes Fusion (WBF) – Multi-Model Ensemble

Combine predictions from multiple models using WBF for better detection coverage.

In [ ]:
from ensemble_boxes import weighted_boxes_fusion

def run_inference(model, img_path, conf=0.15, imgsz=1280):
    """Run YOLO inference and return normalized detections."""
    results = model.predict(img_path, conf=conf, iou=0.6, imgsz=imgsz, device=0, verbose=False)
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    
    boxes, scores, labels = [], [], []
    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            # Normalize to [0, 1]
            boxes.append([x1/w, y1/h, x2/w, y2/h])
            scores.append(float(box.conf[0]))
            labels.append(int(box.cls[0]))
    return boxes, scores, labels, (w, h)

def ensemble_predict(models_dict, img_path, conf=0.10, iou_thr=0.55, skip_box_thr=0.05):
    """Run WBF ensemble across multiple models."""
    all_boxes, all_scores, all_labels = [], [], []
    img_size = None
    
    for name, model in models_dict.items():
        imgsz = 1280 if 'm' in name or 'l' in name else 640
        boxes, scores, labels, size = run_inference(model, img_path, conf=conf, imgsz=imgsz)
        all_boxes.append(boxes if boxes else [[0,0,0,0]])
        all_scores.append(scores if scores else [0])
        all_labels.append(labels if labels else [0])
        img_size = size
    
    # Apply WBF
    weights = [1.0] * len(models_dict)  # equal weights
    fused_boxes, fused_scores, fused_labels = weighted_boxes_fusion(
        all_boxes, all_scores, all_labels,
        weights=weights,
        iou_thr=iou_thr,
        skip_box_thr=skip_box_thr,
    )
    
    # Convert back to absolute coords
    w, h = img_size
    preds = []
    for box, score, label in zip(fused_boxes, fused_scores, fused_labels):
        preds.append({
            'class_id': int(label),
            'bbox': [box[0]*w, box[1]*h, box[2]*w, box[3]*h],
            'score': float(score)
        })
    return preds

print(f"Models available for ensemble: {list(models.keys())}")

In [ ]:
# ── Run ensemble on test set ──
ens_recalls, ens_precisions = [], []
single_recalls, single_precisions = [], []

for img_path in test_images:
    img = cv2.imread(img_path)
    img_h, img_w = img.shape[:2]
    
    # Ground truth
    lbl_name = os.path.splitext(os.path.basename(img_path))[0] + '.txt'
    lbl_path = os.path.join(test_lbl_dir, lbl_name)
    gt = parse_ground_truth(lbl_path, img_w, img_h)
    
    # Single best model
    imgsz = 1280 if 'm' in best_model_name or 'l' in best_model_name else 640
    boxes, scores, labels, _ = run_inference(best_model, img_path, conf=0.15, imgsz=imgsz)
    single_preds = [{'class_id': l, 'bbox': [b[0]*img_w, b[1]*img_h, b[2]*img_w, b[3]*img_h]} 
                    for b, l in zip(boxes, labels)]
    
    # Ensemble
    ens_preds = ensemble_predict(models, img_path, conf=0.10, iou_thr=0.55)
    
    # Compute metrics
    r_single, p_single = compute_recall_precision(single_preds, gt)
    r_ens, p_ens = compute_recall_precision(ens_preds, gt)
    
    single_recalls.append(r_single)
    single_precisions.append(p_single)
    ens_recalls.append(r_ens)
    ens_precisions.append(p_ens)

print(f"\n{'Method':25s} {'Avg Recall':>12s} {'Avg Precision':>14s}")
print(f"{'Single model':25s} {np.mean(single_recalls):12.4f} {np.mean(single_precisions):14.4f}")
print(f"{'WBF Ensemble':25s} {np.mean(ens_recalls):12.4f} {np.mean(ens_precisions):14.4f}")
print(f"\nEnsemble recall improvement: {(np.mean(ens_recalls) - np.mean(single_recalls))*100:+.2f}%")

## 3.5 Combined Pipeline: Best Model + TTA + SAHI + Ensemble

In [ ]:
# ── Final combined comparison ──
results_summary = {
    'Standard (conf=0.15)': (np.mean(std_recalls), np.mean(std_precisions)),
    'SAHI (320px tiles)': (np.mean(sahi_recalls), np.mean(sahi_precisions)),
    'WBF Ensemble': (np.mean(ens_recalls), np.mean(ens_precisions)),
}

fig, ax = plt.subplots(figsize=(10, 6))
methods = list(results_summary.keys())
recalls = [v[0] for v in results_summary.values()]
precisions = [v[1] for v in results_summary.values()]

x = np.arange(len(methods))
w = 0.35
bars1 = ax.bar(x - w/2, recalls, w, label='Recall', color='steelblue')
bars2 = ax.bar(x + w/2, precisions, w, label='Precision', color='coral')

ax.set_ylabel('Metric Value')
ax.set_title('Inference Optimization Comparison')
ax.set_xticks(x)
ax.set_xticklabels(methods, rotation=15)
ax.legend()
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'inference_optimization_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Visualize sample detections ──
def draw_detections(img, preds, class_names, title='', max_draw=100):
    """Draw bounding boxes on image."""
    img_draw = img.copy()
    np.random.seed(42)
    colors = {i: tuple(np.random.randint(50, 255, 3).tolist()) for i in range(len(class_names))}
    
    for i, pred in enumerate(preds[:max_draw]):
        x1, y1, x2, y2 = [int(v) for v in pred['bbox']]
        cls_id = pred['class_id']
        color = colors.get(cls_id, (0, 255, 0))
        cv2.rectangle(img_draw, (x1, y1), (x2, y2), color, 2)
        label = f"{class_names[cls_id]}"
        if 'score' in pred:
            label += f" {pred['score']:.2f}"
        cv2.putText(img_draw, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.35, color, 1)
    return img_draw

# Show 3 sample images with ensemble predictions
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
for idx, img_path in enumerate(test_images[:3]):
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    preds = ensemble_predict(models, img_path, conf=0.10)
    img_draw = draw_detections(img_rgb, preds, class_names)
    
    axes[idx].imshow(img_draw)
    axes[idx].set_title(f'Image {idx+1}: {len(preds)} detections')
    axes[idx].axis('off')

plt.suptitle('Ensemble Detection Results (WBF)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'ensemble_detections.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Final Summary ──
print("\n" + "="*70)
print("📋 INFERENCE OPTIMIZATION SUMMARY")
print("="*70)
print(f"\nBest single model: {best_model_name}")
print(f"Models in ensemble: {list(models.keys())}")
print(f"\nOptimal inference settings:")
print(f"  Confidence threshold: 0.10-0.15")
print(f"  NMS IoU threshold: 0.55-0.60")
print(f"  TTA: enabled (augment=True)")
print(f"  SAHI: 320px tiles with 20% overlap")
print(f"  Ensemble: WBF with iou_thr=0.55")
print(f"\n→ Use these settings for Share of Shelf analysis in Notebook 4")